# InsightOS Qwen3-8B Runtime Evaluation v2.5.0 | Runtime-State Regression

Inference-only evaluation of the unchanged canonical 12-case v4 benchmark using the pinned Qwen3-8B model, current source-derived slim contract, and notebook-local runtime simulation.

**v2.5.0 runtime coverage:** the source snapshot and regression probes cover persisted scheduler attempt/retryability metadata, operation-keyed retry budgets, completed-operation replay protection, final-response enforcement on assembled outcomes, clarification-workflow continuation identity, checkpoint/result retrieval boundaries, and specialist handoff preparation without dispatch.

**Leakage control:** benchmark-derived few-shot examples and gold answers are audited but never placed in model prompts. Synthetic fixtures expose only trusted prior state and verified evidence references, not gold handoffs or decisions.

**Execution boundary:** tools and specialist handoffs are proposal simulations only. No production tools, engines, database mutations, LangGraph tasks, or specialist models are executed. This package is not evidence that model inference has run; a fresh pinned-model Kaggle run and human review are pending. The 12 visible cases are a development-set evaluation, not evidence of generalisation.

The v2.5.0 constrained-decoding and generation-loop hardening is retained. The companion Kaggle inputs ZIP packages hash-verified current runtime source and tests; the canonical benchmark, gold responses, and model pin remain unchanged.

## 1. Environment and reproducibility

The notebook expects a Kaggle GPU runtime with internet access for the pinned public Hugging Face revision. It does not replace PyTorch. The inference configuration follows the established baseline: Qwen3-8B, pinned model/tokenizer commit, NF4 4-bit, FP16 compute, double quantization, greedy decoding and fixed seed.
The response schema is enforced during decoding with the application-generated JSON Schema; the schema object is not placed in the model prompt. This adds a pinned lm-format-enforcer dependency without changing the model or revision.


In [ ]:
%pip install --quiet --upgrade "transformers==4.51.3" "accelerate==1.6.0" "bitsandbytes==0.48.2" "safetensors==0.5.3" "jsonschema==4.26.0" "lm-format-enforcer==0.10.11" 

In [ ]:
from __future__ import annotations
import hashlib, importlib.metadata as metadata, json, os, platform, random, shutil, sys, time, uuid, zipfile
from datetime import datetime, timezone
from pathlib import Path

import torch
from packaging.version import Version

SEED = 20261105
MODEL_ID = "Qwen/Qwen3-8B"
MODEL_REVISION = "b968826d9c46dd6066d109eabc6255188de91218"
MAX_INPUT_TOKENS = 16384
MAX_NEW_TOKENS = 768
GENERATION = {
    "do_sample": False,
    "structured_output": "application JSON Schema constrained decoding",
    "json_schema_constrained_decoding": True,
    "enable_thinking": False,
    "max_input_tokens": MAX_INPUT_TOKENS,
    "max_new_tokens": MAX_NEW_TOKENS,
    "seed": SEED,
    "temperature": None,
    "top_p": None,
    "tool_execution": False,
    "generation_hardening": "v2.4.0 constrained-decoding and generation-loop hardening retained; v2.5.0 adds runtime-state regression probes",
}

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
try:
    torch.use_deterministic_algorithms(True, warn_only=True)
except (AttributeError, TypeError):
    pass

if not torch.cuda.is_available():
    raise RuntimeError("A Kaggle GPU with CUDA is required; enable GPU in notebook settings.")
if not torch.version.cuda:
    raise RuntimeError("The installed PyTorch is not a CUDA build.")
gpu = torch.cuda.get_device_properties(0)
gpu_memory_gib = gpu.total_memory / (1024 ** 3)
if gpu_memory_gib < 12:
    raise RuntimeError(f"GPU memory is {gpu_memory_gib:.1f} GiB; use a Kaggle GPU with at least 12 GiB.")

required_versions = {
    "transformers": "4.51.3",
    "accelerate": "1.6.0",
    "bitsandbytes": "0.48.2",
    "safetensors": "0.5.3",
    "jsonschema": "4.26.0",
    "lm-format-enforcer": "0.10.11",
}
versions = {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda}
if not Version("2.6.0") <= Version(torch.__version__.split("+", 1)[0]) < Version("3.0.0"):
    raise RuntimeError(f"PyTorch {torch.__version__} is outside the required >=2.6.0,<3.0.0 range.")
for package, expected in required_versions.items():
    actual = metadata.version(package)
    versions[package] = actual
    if actual != expected:
        raise RuntimeError(f"{package} version {actual} does not match required {expected}; restart and rerun setup.")
try:
    pydantic_version = metadata.version("pydantic")
except metadata.PackageNotFoundError as error:
    raise RuntimeError("Pydantic v2 is required to load the exact slim response model.") from error
versions["pydantic"] = pydantic_version
if int(pydantic_version.split(".", 1)[0]) != 2:
    raise RuntimeError(f"Pydantic v2 is required; found {pydantic_version}.")

disk = shutil.disk_usage("/kaggle/working")
with open("/proc/meminfo", encoding="utf-8") as memory_file:
    memory_values = {
        line.split(":", 1)[0]: int(line.split()[1]) * 1024
        for line in memory_file
        if ":" in line
    }
environment = {
    "kaggle": Path("/kaggle").exists(),
    "python": platform.python_version(),
    "packages": versions,
    "gpu_name": gpu.name,
    "gpu_memory_gib": round(gpu_memory_gib, 3),
    "gpu_capability": list(torch.cuda.get_device_capability(0)),
    "ram_total_gib": round(memory_values["MemTotal"] / (1024 ** 3), 3),
    "ram_available_gib": round(memory_values.get("MemAvailable", 0) / (1024 ** 3), 3),
    "working_disk_free_gib": round(disk.free / (1024 ** 3), 3),
    "huggingface_cache_dir": "/kaggle/working/hf_cache",
    "generation": GENERATION,
}
if not environment["kaggle"]:
    raise RuntimeError("Run this notebook in Kaggle; local paths and hardware are not assumed.")
if memory_values.get("MemTotal", 0) < 8 * (1024 ** 3):
    raise RuntimeError("At least 8 GiB system RAM is required for model loading.")
# Dynamic storage policy: check the actual pinned repository size before download.
MIN_FREE_BEFORE_MODEL_PREFLIGHT_GIB = 2.0
if disk.free < MIN_FREE_BEFORE_MODEL_PREFLIGHT_GIB * (1024 ** 3):
    raise RuntimeError(
        f"Only {disk.free / (1024 ** 3):.2f} GiB is free. "
        f"At least {MIN_FREE_BEFORE_MODEL_PREFLIGHT_GIB:.1f} GiB is required "
        "to stage inputs and perform the repository-size preflight."
    )
if gpu_memory_gib < 16:
    print(f"Warning: {gpu.name} has {gpu_memory_gib:.1f} GiB VRAM; 16 GiB/T4-or-better is recommended.")
print(json.dumps(environment, indent=2))

## 2. Input artifacts and integrity verification

The companion mounted Kaggle Dataset directory contains only the benchmark bundle, canonical benchmark/gold/few-shot files, and source files needed for inspection. The ZIP is extracted only after checking each member path and validating the embedded file-hash manifest. Missing, duplicate, altered, or mismatched inputs stop the run; no synthetic replacements are created.

In [ ]:
EXPECTED_INPUT_HASHES = {'README.md': '745b9b440a2720d0db9a0f96ea6b406e250676363fb2b9fe2021f39f261c4f21',
 'application/app/agent/api.py': '318948657d91b3389beab516ad2fb5bcf240e2abfff086c176bfeb9ffd8110f0',
 'application/app/agent/business_intelligence.py': 'fb6cba8e0280f9c79c91e16986d0a93000915630aa337e177423909a90b6b34e',
 'application/app/agent/checkpointer.py': 'ccf2ec359e834916de9aaac4c353a90c60ad08482a1f3c6e98593dd8683aee07',
 'application/app/agent/config.py': '38a9688af2b93cf806e1cb6b602b52207ed8772180cf1ab42a8a6388f9878a6f',
 'application/app/agent/contracts.py': '4eee4a40badd461e8820128650a63fba18b637465caf58af33d0af4135f0f8dd',
 'application/app/agent/langgraph_orchestrator.py': 'f5c150040ba65b96c7f31d3a2b0feb595ba563773ad155af3ce1665f0c486657',
 'application/app/agent/model_registry.py': '34699cfd3b86dcd7204c96acc0f4359efe2a4b40e551c95591aa47c86f770980',
 'application/app/agent/orchestrator.py': '56788c672a2ba10ff88882576d8164734a2758675e3a565eea2b424a794cc9db',
 'application/app/agent/orchestrator_few_shot_v1.json': '5d5a764f3189aff69247001a55eb921713e575fb6dfdbffc1c024954e99bcbae',
 'application/app/agent/providers.py': 'bb717ced56405ec6806151fa5f29a6d4d64fc70c6a2666eb05174fd2380e5898',
 'application/app/agent/runtime.py': 'a8ad4ac3fe40e6f6d20964643f0a4a4f2c0a72c87c11549ed57f4b9d301c7f32',
 'application/app/agent/slim_orchestrator.py': 'f08d9576d2b1b3c830d520880ddef68c2de9e64c27ece86a0ab23bfa5903b942',
 'application/app/agent/specialist_agents.py': '2f2b4676a9b41c9038ff5598232baf9adef7aa05ae38304e064381e39ec659e9',
 'application/app/agent/tools.py': '694336a332a3cae6cc435368537abff72dcfd13102325447ade38239f8f76a4b',
 'application/app/workflow_execution.py': '9fc1e9e838662098c9fde40af9b70a157775152893dae5932839cb3cea42864c',
 'application/tests/test_agent_api.py': 'f6f651146bba30c41a307ce9c242a50eab21b9bf3c55404e86e27f3bcd20d868',
 'application/tests/test_agent_runtime.py': '6afb8a18af4dd303cb01d6d06e1e49b0bd67da1073fb3b975ca9a1717645b223',
 'application/tests/test_langgraph_milestone1.py': '497eb287af2673d093803e76a49f71a9353dd31f65600078d1eaa70eaa27ab46',
 'application/tests/test_langgraph_orchestrator.py': 'c3fb33487f6a3a1898b9965dc75706b60e00e254c38a44d8980120a91c9092eb',
 'application/tests/test_langgraph_postgres_integration.py': 'c0cf8feb5f05e37f6802d051e30dad557d15e1f3b8b7efc316b3b17c7de6dcc1',
 'application/tests/test_retry_and_length_policy.py': 'dd8a74bf2f3affb983dd946bb7a3ebed6469335de218d1ce8cd8ab5eb07a1467',
 'application/tests/test_runtime_corrections.py': '30bfa3fc758a6ee8b253b60b72ee38ee196155d5ab8203b2df519c60573c0194',
 'application/tests/test_slim_orchestrator.py': '73df1db6008859ec7a3f60fc8d43429808b40c6b93eb9441f442f4828b3e592c',
 'application/tests/test_workflow_execution.py': 'ba11a3f833435cb96711748ac91496ee8230e74f73d158537bca11c219eec12c',
 'benchmark/qwen3-8b-baseline-bundle-v6.json': '3c4c24d4b040e1b8c4fd515ff0ecb8af8b9d8dd18dd2fd13179a5c7b463a9cf9',
 'benchmark/qwen3-8b-gold-responses-v1.json': '62112cd56975f335e8016a6eb6c6099320a18439e1f84cfae4f76c9d3a20ab7d',
 'benchmark/qwen3-8b-orchestration-benchmark-v4.json': '8c68410f14336956051461b585980cf5cda905a9a2852321be842ef16ae04a42'}
EXPECTED_INPUT_MANIFEST_SHA256 = "402cf9378f208128d7bfe8ba87ff83c4645eed741aa059b4ecea4089466fede2"
# Kaggle Dataset is mounted as a directory. Use the exact mounted root supplied
# by the user; do not search for or extract a ZIP.
INPUT_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/nithinchandrab/insightos-qwen3-8b-runtime-evaluation-2-5-0"),
    Path("/kaggle/input/insightos-qwen3-8b-runtime-evaluation-2-5-0"),
]
# Every file and the manifest are pinned; stale or altered input bundles fail loudly.
INPUT_ROOT = next((path for path in INPUT_ROOT_CANDIDATES if path.is_dir()), INPUT_ROOT_CANDIDATES[0])

if not INPUT_ROOT.is_dir():
    raise FileNotFoundError(
        "Kaggle input dataset directory was not found: "
        f"{INPUT_ROOT}. Confirm the dataset is attached to this notebook."
    )

def sha256_bytes(value):
    return hashlib.sha256(value).hexdigest()

# Check the required directory structure first and report every missing path
# in one error, rather than failing on the first file.
missing_inputs = [
    relative_path for relative_path in EXPECTED_INPUT_HASHES
    if not (INPUT_ROOT / relative_path).is_file()
]
if missing_inputs:
    raise FileNotFoundError(
        "The attached Kaggle dataset is missing these required files:\\n"
        + "\\n".join(f" - {relative_path}" for relative_path in missing_inputs)
        + f"\\nDataset root checked: {INPUT_ROOT}"
    )

for relative_path, expected_hash in EXPECTED_INPUT_HASHES.items():
    source_path = INPUT_ROOT / relative_path
    if not source_path.is_file():
        raise FileNotFoundError(f"Required source/artifact is missing: {relative_path}")
    actual_hash = sha256_bytes(source_path.read_bytes())
    if actual_hash != expected_hash:
        raise RuntimeError(f"SHA-256 mismatch for {relative_path}: {actual_hash}")

manifest_path = INPUT_ROOT / "input-manifest.json"
if not manifest_path.is_file() or sha256_bytes(manifest_path.read_bytes()) != EXPECTED_INPUT_MANIFEST_SHA256:
    raise RuntimeError("The companion input manifest is missing or has an unexpected SHA-256.")
input_manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
if input_manifest.get("files") != EXPECTED_INPUT_HASHES:
    raise RuntimeError("The input manifest file list differs from the notebook-pinned hashes.")

print("Using mounted Kaggle dataset root:", INPUT_ROOT)
print("Verified required input files:", len(EXPECTED_INPUT_HASHES))

bundle_path = INPUT_ROOT / "benchmark/qwen3-8b-baseline-bundle-v6.json"
benchmark_path = INPUT_ROOT / "benchmark/qwen3-8b-orchestration-benchmark-v4.json"
gold_path = INPUT_ROOT / "benchmark/qwen3-8b-gold-responses-v1.json"
few_shot_path = INPUT_ROOT / "application/app/agent/orchestrator_few_shot_v1.json"
bundle = json.loads(bundle_path.read_text(encoding="utf-8"))
benchmark = json.loads(benchmark_path.read_text(encoding="utf-8"))
gold = json.loads(gold_path.read_text(encoding="utf-8"))
few_shot_payload = json.loads(few_shot_path.read_text(encoding="utf-8"))
canonical = lambda obj: json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False).encode("utf-8")
if benchmark != bundle.get("benchmark"):
    raise RuntimeError("Standalone benchmark differs from the benchmark embedded in the v6 bundle.")
if sha256_bytes(canonical(benchmark)) != bundle.get("benchmark_sha256"):
    raise RuntimeError("Canonical benchmark hash does not match the bundle metadata.")
if sha256_bytes(canonical(bundle["structured_output_contracts"])) != bundle.get("structured_output_contracts_sha256"):
    raise RuntimeError("Structured contract hash does not match the bundle metadata.")
tool_schemas_sorted = sorted(bundle["tool_schemas"], key=lambda item: item["function"]["name"])
if sha256_bytes(canonical(tool_schemas_sorted)) != bundle.get("tool_schemas_sha256"):
    raise RuntimeError("Tool schema snapshot hash does not match the bundle metadata.")

cases = benchmark.get("cases")
case_ids = [case.get("id") for case in cases] if isinstance(cases, list) else []
if benchmark.get("benchmark_version") != "insightos-qwen3-8b-orchestration-benchmark-v4":
    raise RuntimeError("Unexpected benchmark version.")
if len(cases) != 12 or len(set(case_ids)) != 12 or any(not isinstance(case_id, str) for case_id in case_ids):
    raise RuntimeError("Expected exactly 12 unique benchmark case IDs.")
if any(case.get("split") != "independent_test" for case in cases):
    raise RuntimeError("Benchmark contains a case outside the independent test split.")
if bundle.get("model", {}).get("revision") != MODEL_REVISION:
    raise RuntimeError("Model revision differs from the baseline bundle.")
if few_shot_payload.get("source_sha256") != EXPECTED_INPUT_HASHES["benchmark/qwen3-8b-gold-responses-v1.json"]:
    raise RuntimeError("Few-shot bundle does not reference the canonical gold SHA-256.")
if few_shot_payload.get("benchmark_version") != benchmark["benchmark_version"]:
    raise RuntimeError("Few-shot bundle benchmark version differs.")
if len(few_shot_payload.get("examples", [])) != 12:
    raise RuntimeError("Few-shot bundle must contain all 12 canonical examples.")
if set(gold) != set(case_ids):
    raise RuntimeError("Gold responses do not cover the benchmark case IDs exactly.")
few_shot_by_id = {item["case_id"]: item for item in few_shot_payload["examples"]}
for case_id, source_gold in gold.items():
    source_handoff = source_gold["specialist_handoff"]
    expected_slim_response = {
        "decision": source_gold["decision"],
        "intent": source_gold["intent"],
        "plan": source_gold["plan"],
        "tool_calls": source_gold["tool_calls"],
        "handoff_request": (
            {"agent": source_handoff["agent"], "objective": source_handoff["objective"]}
            if source_handoff is not None
            else None
        ),
        "response": source_gold["response"],
        "clarification_questions": source_gold["clarification_questions"],
    }
    if few_shot_by_id[case_id]["prompt"] != next(case["prompt"] for case in cases if case["id"] == case_id):
        raise RuntimeError(f"Few-shot prompt differs from benchmark case {case_id}.")
    if few_shot_by_id[case_id]["response"] != expected_slim_response:
        raise RuntimeError(f"Few-shot example differs from canonical gold for {case_id}.")
print(f"Verified {len(cases)} cases, {len(bundle['tool_schemas'])} source-snapshot tool schemas, and all input hashes.")

In [ ]:
import ast, importlib.util, re, sys, traceback
from jsonschema import Draft202012Validator

APP_ROOT = INPUT_ROOT / "application/app/agent"
SLIM_PATH = APP_ROOT / "slim_orchestrator.py"
RUNTIME_PATH = APP_ROOT / "runtime.py"
BI_PATH = APP_ROOT / "business_intelligence.py"
CONTRACTS_PATH = APP_ROOT / "contracts.py"

def import_source_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Cannot load source module {path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

slim = import_source_module("insightos_slim_orchestrator_source", SLIM_PATH)
bi = import_source_module("insightos_business_intelligence_source", BI_PATH)
contracts = import_source_module("insightos_contracts_source", CONTRACTS_PATH)
runtime_source = RUNTIME_PATH.read_text(encoding="utf-8")
runtime_tree = ast.parse(runtime_source, filename=str(RUNTIME_PATH))

def compile_source_functions(function_names, globals_dict):
    selected = [node for node in runtime_tree.body if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name in function_names]
    missing = set(function_names) - {node.name for node in selected}
    if missing:
        raise RuntimeError(f"Expected runtime helper(s) not found: {sorted(missing)}")
    module = ast.Module(body=[ast.ImportFrom(module="__future__", names=[ast.alias(name="annotations")], level=0), *selected], type_ignores=[])
    exec(compile(ast.fix_missing_locations(module), str(RUNTIME_PATH), "exec"), globals_dict)
    return globals_dict

system_node = next(
    node for node in runtime_tree.body
    if isinstance(node, ast.Assign) and any(isinstance(target, ast.Name) and target.id == "SYSTEM_INSTRUCTIONS" for target in node.targets)
)
SYSTEM_INSTRUCTIONS = ast.literal_eval(system_node.value)
runtime_helpers = compile_source_functions(
    {"_orchestrator_system_instructions", "_slim_contract_prompt", "_prior_verified_profile", "_trusted_runtime_state", "_numeric_evidence_facts", "_completed_engine_operation", "_runtime_state_sets", "_pending_clarification_from_context", "_protected_workflow_ids", "_short_response_piece", "_safe_response_fallback", "_enforce_final_response_policy"},
    {
        "json": json,
        "re": re,
        "SYSTEM_INSTRUCTIONS": SYSTEM_INSTRUCTIONS,
        "SPECIALIST_PROMPT_PROFILES": bi.SPECIALIST_PROMPT_PROFILES,
        "model_specialization_instruction": bi.model_specialization_instruction,
        "SLIM_ORCHESTRATOR_INSTRUCTIONS": slim.SLIM_ORCHESTRATOR_INSTRUCTIONS,
        "response_json_schema": slim.response_json_schema,
        "_ENGINE_OPERATION_TO_RESULT_RETRIEVAL": {"run_eda_analysis": "get_eda_result"},
        "build_tool_retry_state": slim.build_tool_retry_state,
        "ENGINE_OPERATION_TOOLS": slim.ENGINE_OPERATION_TOOLS,
        "AgentOutcome": contracts.AgentOutcome,
        "replace": __import__("dataclasses").replace,
        "split_sentences": slim.split_sentences,
        "final_response_length_issue": slim.final_response_length_issue,
    },
)
system_prompt = runtime_helpers["_orchestrator_system_instructions"](MODEL_ID.replace("Qwen/Qwen3-8B", "qwen3:8b"))
if not isinstance(system_prompt, str) or not system_prompt.startswith(SYSTEM_INSTRUCTIONS):
    raise RuntimeError("Could not assemble the exact source-derived orchestrator system prompt.")

# Audit the bundled production examples, but NEVER place them in evaluation prompts.
bundled_production_examples = slim.load_few_shot_examples()
benchmark_prompts = {case["prompt"] for case in cases}
overlapping_examples = [
    example for example in bundled_production_examples
    if example.get("prompt") in benchmark_prompts
]
if len(overlapping_examples) != len(cases):
    raise RuntimeError(
        "Expected to detect the known benchmark/few-shot overlap; inspect the input bundle."
    )
few_shot_examples = []
source_prompt_probe = runtime_helpers["_slim_contract_prompt"](
    (), runtime_helpers["_trusted_runtime_state"]([], [])
)
if "few_shot_examples" in source_prompt_probe:
    raise RuntimeError("Leakage guard failed: source-derived prompt contains few-shot examples.")

tool_schema_by_name = {
    item["function"]["name"]: item
    for item in bundle["tool_schemas"]
}
def case_tool_schemas(case):
    names = case.get("tools", [])
    missing = sorted(set(names) - set(tool_schema_by_name))
    if missing:
        raise RuntimeError(f"Case {case['id']} names tools absent from source snapshot: {missing}")
    return tuple(tool_schema_by_name[name] for name in names)

for case in cases:
    slim.response_json_schema(case_tool_schemas(case))
for source_path in APP_ROOT.rglob("*"):
    if source_path.is_symlink() and not source_path.resolve(strict=True).exists():
        raise RuntimeError(f"Broken source package symlink: {source_path}")
print("Loaded source-derived slim model, policy validator, current source prompts and runtime prompt helpers.")
print("Bundled production examples audited:", len(bundled_production_examples), "| overlapping test examples excluded:", len(overlapping_examples))
print("Few-shot examples passed to evaluation model:", len(few_shot_examples))
print("System prompt characters:", len(system_prompt))

## 3. Current model-facing contract and leakage-free prompt

The effective system instructions, slim response schema, policy validator and runtime prompt builder are derived from the refreshed application source snapshot.

The production prompt builder no longer inserts the benchmark-derived few-shot artifact. The notebook audits the artifact for overlap, but never passes its examples or gold answers to the model. Assertions below reject benchmark prompts, gold answers, or a few-shot field in the source-derived prompt.

The model receives the current compact source-derived slim contract prompt, available tool schemas, evidence-provenance categories, the case prompt, and explicitly marked synthetic prior-result fixtures where applicable. The complete application-generated response JSON Schema is used only by the constrained decoder and evaluator; it is not copied into the prompt. Evaluator-only expected behavior and specialist-input contracts remain out of model context.


In [ ]:
tool_schema_by_name_for_case = {case["id"]: case_tool_schemas(case) for case in cases}
example_case = cases[0]
example_runtime_state = runtime_helpers["_trusted_runtime_state"]([], [])
example_contract_prompt = runtime_helpers["_slim_contract_prompt"](
    tool_schema_by_name_for_case[example_case["id"]],
    example_runtime_state,
)
if any(case["prompt"] in example_contract_prompt for case in cases):
    raise RuntimeError("Benchmark prompt leakage detected in the model-facing contract prompt.")
if "few_shot_examples" in example_contract_prompt:
    raise RuntimeError("Model-facing prompt unexpectedly contains a few-shot field.")
for case in cases:
    gold_response = json.dumps(gold[case["id"]], ensure_ascii=False, sort_keys=True)
    if gold_response in example_contract_prompt:
        raise RuntimeError("Canonical gold response leaked into the model-facing prompt.")
print("=== Exact source-derived system prompt ===")
print(system_prompt)
print("\\n=== Updated source-derived slim contract prompt for the first case ===")
print(example_contract_prompt)


## 3b. Retained generation hardening (v2.4.0)

The decoder uses a **stricter, length-bounded copy** of the source response schema, so any accepted output still validates against the unchanged source `SlimOrchestratorTurn` and policy validator. The added system message is generic (no case IDs, prompts or gold text) and mirrors rules the source validator already enforces; the self-test below asserts it contains no benchmark prompt or gold response.

In [ ]:
# ---------------------------------------------------------------------------
# Retained v2.4.0 generation hardening (notebook-local; production source is untouched)
#
# Root cause of the three v2.3.0 failures: the source response schema lets
# `response` grow to 8,000 characters (~2,000 tokens) while generation is capped
# at 512 tokens. Under greedy decoding the model never closes the string; it
# loops on closing boilerplate ("If you have any other requests ...") until the
# token budget is gone. The structured fields (decision / tool_calls /
# handoff_request) were already correct in attempt 1 for all three cases.
# The repair prompt then echoed the looping text back, which re-poisoned
# attempts 2 and 3.
# ---------------------------------------------------------------------------
import copy, json, re

GENERATION_CAPS = {
    "intent_max_chars": 160,
    "plan_max_items": 4,
    "plan_item_max_chars": 200,
    "response_max_chars": 500,
    "clarification_max_items": 3,
    "clarification_item_max_chars": 200,
    "handoff_objective_max_chars": 300,
}


def _resolve(schema, node):
    """Follow a local $ref once so the cap lands on the real definition."""
    ref = node.get("$ref") if isinstance(node, dict) else None
    if ref and ref.startswith("#/$defs/"):
        return schema.get("$defs", {}).get(ref.split("/")[-1], node)
    return node


def _cap_string(node, max_chars):
    if isinstance(node, dict) and node.get("type") == "string":
        node["maxLength"] = min(int(node.get("maxLength", max_chars)), max_chars)
        return True
    return False


def tighten_generation_schema(source_schema, caps=GENERATION_CAPS):
    """Return a *stricter* copy of the source schema used only for decoding.

    Every output that satisfies the tightened schema also satisfies the source
    schema / SlimOrchestratorTurn, so runtime validation is unchanged.
    """
    schema = copy.deepcopy(source_schema)
    props = schema["properties"]

    _cap_string(props["intent"], caps["intent_max_chars"])
    _cap_string(props["response"], caps["response_max_chars"])

    plan = props["plan"]
    plan["maxItems"] = min(int(plan.get("maxItems", caps["plan_max_items"])), caps["plan_max_items"])
    _cap_string(plan.setdefault("items", {"type": "string"}), caps["plan_item_max_chars"])

    clar = props["clarification_questions"]
    clar["maxItems"] = min(int(clar.get("maxItems", caps["clarification_max_items"])), caps["clarification_max_items"])
    _cap_string(clar.setdefault("items", {"type": "string"}), caps["clarification_item_max_chars"])

    # handoff_request is usually anyOf[{$ref: Handoff}, {type: null}]
    handoff = props["handoff_request"]
    branches = handoff.get("anyOf", [handoff])
    for branch in branches:
        target = _resolve(schema, branch)
        objective = target.get("properties", {}).get("objective") if isinstance(target, dict) else None
        if objective is not None:
            _cap_string(objective, caps["handoff_objective_max_chars"])
    return schema


# Generic response-style contract. It names no benchmark case, prompt or answer.
GENERATION_BUDGET_ADDENDUM = "\n".join((
    "Output budget (runtime-enforced): keep `response` to at most two short sentences "
    f"(under {GENERATION_CAPS['response_max_chars']} characters). State only facts present in the request "
    "or trusted context. Never end with an offer of further help, never repeat a sentence, "
    "and stop as soon as the JSON object is complete.",
    "A tool call or handoff is only a proposal: describe it as proposed or pending, "
    "never as already run, retrieved, prepared or dispatched.",
    'When the user supplies workspace profile or result facts and asks what follows from them, '
    'use decision "continue" with no tool call, attribute each number to its source with wording such as '
    '"you reported" or "the current profile you provided shows", add no number that is not in the request, '
    'and add no qualitative judgment (small, large, complete, clean, representative) that the user did not make.',
))


def has_repetition_loop(token_ids, window=16, repeats=3):
    """True when the tail is the same `window`-token block repeated `repeats` times."""
    n = window * repeats
    if len(token_ids) < n:
        return False
    tail = list(token_ids[-n:])
    block = tail[-window:]
    return all(tail[i * window:(i + 1) * window] == block for i in range(repeats))


def make_repetition_stopping_criteria():
    from transformers import StoppingCriteria

    class RepetitionLoopStop(StoppingCriteria):
        def __init__(self):
            self.triggered = False

        def __call__(self, input_ids, scores, **kwargs):
            if has_repetition_loop(input_ids[0].tolist()):
                self.triggered = True
                return True
            return False

    return RepetitionLoopStop()


def repair_message_v2(raw_text, issue, degenerate):
    """Never echo degenerate (looping/truncated) text back into the context."""
    keys = "decision, intent, plan, tool_calls, handoff_request, response, clarification_questions"
    if degenerate:
        return [{"role": "user", "content": (
            f"Your previous reply was discarded because it did not terminate ({issue}). "
            f"Return one complete JSON object with exactly these top-level keys: {keys}. "
            f"Keep `response` to one or two short sentences, do not repeat yourself, "
            "and do not offer further help.")}]
    return [
        {"role": "assistant", "content": raw_text[:1500]},
        {"role": "user", "content": (
            f"The previous response failed runtime contract validation ({issue}). "
            f"Return one corrected complete JSON object with exactly these top-level keys: {keys}. "
            "Do not include schema definitions, extra fields, an invalid proposal, or a claim that an action executed.")},
    ]


# --- negation / conditional aware causal-claim heuristic --------------------
CAUSAL_CLAIM = re.compile(
    r"\b(?:caused by|(?:campaign|promotion|change|it|this)\s+(?:may\s+have\s+|has\s+|have\s+)?caused|"
    r"proves? causation|causally attributable|"
    r"(?:may|might|could|possibly|potentially|likely|probably)\s+(?:have\s+)?"
    r"(?:caused|contributed|led|driven|influenced|resulted in))\b",
    re.IGNORECASE,
)
CAUSAL_GUARD = re.compile(
    r"\b(?:not|no|cannot|can't|isn't|doesn't|does not|do not|without|unless|whether|if|"
    r"to (?:determine|establish|test|assess|show)|required|requires|need(?:s|ed)?|would)\b",
    re.IGNORECASE,
)


def unsupported_causal_claim(text):
    """Flag a sentence only when it asserts causation without negation/conditioning."""
    for sentence in re.split(r"(?<=[.!?])\s+|\n", text or ""):
        if CAUSAL_CLAIM.search(sentence) and not CAUSAL_GUARD.search(sentence):
            return True
    return False


# --- self-tests: run BEFORE the model download so a mistake fails fast -------
from jsonschema import Draft202012Validator as _V
for _case in cases:
    _src = slim.response_json_schema(case_tool_schemas(_case))
    _tgt = tighten_generation_schema(_src)
    _V.check_schema(_tgt)
    assert _tgt["properties"]["response"]["maxLength"] <= GENERATION_CAPS["response_max_chars"]
    assert _tgt["properties"]["plan"]["maxItems"] <= GENERATION_CAPS["plan_max_items"]
assert has_repetition_loop([1, 2, 3] + [7] * 16 * 3) and not has_repetition_loop(list(range(200)))
assert unsupported_causal_claim("The campaign caused the sales increase.")
assert not unsupported_causal_claim("Correlation does not establish whether the campaign caused the increase.")
assert not any(c["prompt"] in GENERATION_BUDGET_ADDENDUM for c in cases), "benchmark prompt leaked into addendum"
assert not any(json.dumps(gold[c["id"]]["response"]) in GENERATION_BUDGET_ADDENDUM for c in cases)
print("Hardening self-tests passed for", len(cases), "cases.")


## 4. Model loading

The Hugging Face API commit and the downloaded snapshot path must match the pinned revision. The model and tokenizer load from the same verified local snapshot. Quantization or device errors are fatal; this notebook will not silently fall back to FP16/BF16 weights or CPU.

In [ ]:
# Use one explicit Hugging Face cache in the Kaggle working volume.
HF_HOME = "/kaggle/working/hf_cache"
HF_HUB_CACHE = f"{HF_HOME}/hub"
os.environ["HF_HOME"] = HF_HOME
os.environ["HF_HUB_CACHE"] = HF_HUB_CACHE
os.environ["TRANSFORMERS_CACHE"] = f"{HF_HOME}/transformers"

from huggingface_hub import HfApi, snapshot_download
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from lmformatenforcer.integrations.transformers import build_token_enforcer_tokenizer_data
import bitsandbytes as bnb
from bitsandbytes import cuda_specs, cextension

native_library = getattr(cextension, "lib", None)
if not bool(getattr(native_library, "compiled_with_cuda", False)):
    raise RuntimeError("bitsandbytes loaded a CPU-only binary; a CUDA-enabled binary is required.")
cuda_spec = cuda_specs.get_cuda_specs()
if cuda_spec is None:
    raise RuntimeError("bitsandbytes could not determine the active CUDA binary specification.")
bnb_cuda_path = cextension.get_cuda_bnb_library_path(cuda_spec)
if bnb_cuda_path is None or not bnb_cuda_path.is_file():
    raise RuntimeError(f"bitsandbytes CUDA binary is missing for {torch.version.cuda}: {bnb_cuda_path}")
try:
    probe = torch.randn(128, device="cuda", dtype=torch.float16)
    packed, quant_state = bnb.functional.quantize_4bit(probe, compress_statistics=True, quant_type="nf4")
    restored = bnb.functional.dequantize_4bit(packed, quant_state=quant_state)
    torch.cuda.synchronize()
    if packed.device.type != "cuda" or restored.device.type != "cuda":
        raise RuntimeError("NF4 probe returned a non-CUDA tensor.")
    if not torch.isfinite(packed).all().item() or not torch.isfinite(restored).all().item():
        raise RuntimeError("NF4 probe produced non-finite values.")
except Exception as error:
    raise RuntimeError(f"NF4 CUDA preflight failed: {type(error).__name__}: {error}") from error
cuda_preflight = {
    "status": "passed",
    "torch_cuda": torch.version.cuda,
    "bitsandbytes_cuda_binary": str(bnb_cuda_path),
    "compiled_with_cuda": True,
    "nf4_quantize_dequantize": "passed",
}

started = time.perf_counter()
hf_api = HfApi()
hub_info = hf_api.model_info(MODEL_ID, revision=MODEL_REVISION, files_metadata=True)
if hub_info.sha != MODEL_REVISION:
    raise RuntimeError(f"Hugging Face resolved {hub_info.sha}, expected {MODEL_REVISION}.")

# Estimate the exact pinned repository size before downloading any model files.
repo_files = list(getattr(hub_info, "siblings", []) or [])
reported_sizes = [getattr(item, "size", None) for item in repo_files]
if not repo_files or any(size is None for size in reported_sizes):
    raise RuntimeError(
        "Hugging Face did not return complete repository file sizes. "
        "Cannot safely estimate Kaggle storage; model download was not started."
    )
snapshot_bytes_estimate = sum(int(size) for size in reported_sizes)
snapshot_gib_estimate = snapshot_bytes_estimate / (1024 ** 3)
MODEL_DOWNLOAD_RESERVE_GIB = 2.0
disk_before_download = shutil.disk_usage("/kaggle/working")
required_free_bytes = snapshot_bytes_estimate + int(MODEL_DOWNLOAD_RESERVE_GIB * (1024 ** 3))
if disk_before_download.free < required_free_bytes:
    raise RuntimeError(
        f"Insufficient storage for the pinned model snapshot. "
        f"Snapshot estimate: {snapshot_gib_estimate:.2f} GiB; "
        f"required with {MODEL_DOWNLOAD_RESERVE_GIB:.1f} GiB reserve: "
        f"{required_free_bytes / (1024 ** 3):.2f} GiB; "
        f"currently free: {disk_before_download.free / (1024 ** 3):.2f} GiB. "
        "No model download was started. Use a Kaggle session with more working storage."
    )

snapshot_path = Path(snapshot_download(
    MODEL_ID,
    revision=MODEL_REVISION,
    cache_dir=HF_HUB_CACHE,
    resume_download=True,
)).resolve()
if MODEL_REVISION not in snapshot_path.parts:
    raise RuntimeError(f"Resolved snapshot path does not identify pinned commit: {snapshot_path}")
required_snapshot_files = ["config.json", "tokenizer_config.json"]
for filename in required_snapshot_files:
    if not (snapshot_path / filename).is_file():
        raise FileNotFoundError(f"Pinned snapshot lacks {filename}")
if not any((snapshot_path / name).is_file() for name in ("tokenizer.json", "tokenizer.model", "vocab.json", "spiece.model")):
    raise FileNotFoundError("Pinned snapshot has no tokenizer vocabulary.")
index_names = ("model.safetensors.index.json", "pytorch_model.bin.index.json")
weight_index = next((snapshot_path / name for name in index_names if (snapshot_path / name).is_file()), None)
if weight_index:
    index = json.loads(weight_index.read_text(encoding="utf-8"))
    missing_shards = sorted({name for name in index.get("weight_map", {}).values() if not (snapshot_path / name).is_file()})
    if not index.get("weight_map") or missing_shards:
        raise RuntimeError(f"Model snapshot is incomplete; missing shards: {missing_shards}")
elif not any((snapshot_path / name).is_file() for name in ("model.safetensors", "pytorch_model.bin")):
    raise FileNotFoundError("Pinned snapshot has no model weight files.")

tokenizer = AutoTokenizer.from_pretrained(snapshot_path, local_files_only=True, trust_remote_code=False)
constraint_setup_started = time.perf_counter()
TOKENIZER_ENFORCER_DATA = build_token_enforcer_tokenizer_data(tokenizer)
constraint_setup_seconds = time.perf_counter() - constraint_setup_started
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    snapshot_path,
    local_files_only=True,
    trust_remote_code=False,
    quantization_config=quantization_config,
    device_map={"": 0},
    torch_dtype=torch.float16,
)
model.eval()
torch.cuda.reset_peak_memory_stats()
model_load_seconds = time.perf_counter() - started
print(json.dumps({
    "model": MODEL_ID,
    "revision": MODEL_REVISION,
    "tokenizer": tokenizer.name_or_path,
    "snapshot_path": str(snapshot_path),
    "snapshot_size_estimate_gib": round(snapshot_gib_estimate, 3),
    "hf_cache_dir": HF_HOME,
    "working_disk_free_after_download_gib": round(shutil.disk_usage("/kaggle/working").free / (1024 ** 3), 3),
    "quantization": {"load_in_4bit": True, "type": "nf4", "compute_dtype": "float16", "double_quant": True},
    "cuda_preflight": cuda_preflight,
    "model_load_seconds": round(model_load_seconds, 3),
    "json_schema_constraint_setup_seconds": round(constraint_setup_seconds, 3),
    "allocated_gib": round(torch.cuda.memory_allocated() / (1024 ** 3), 3),
    "peak_allocated_gib": round(torch.cuda.max_memory_allocated() / (1024 ** 3), 3),
}, indent=2))

## 5. Runtime-aware evaluation adapter (simulated)

The adapter reuses the source `SlimOrchestratorTurn`, policy validator, retry identity/state builder, response-length policy, clarification workflow helpers, and trusted-state extraction. Runtime-produced IDs and dispatch decisions remain outside model output. Each benchmark case records prepared proposals and final assembled text; no production app, tool, engine, database mutation, checkpoint task, or specialist is executed. A separate set of source-derived probes exercises scheduler retry states, clarification continuation, completed checkpoint replay rejection, final-response enforcement, and handoff non-dispatch.

In [ ]:
import csv, io, statistics
from jsonschema import ValidationError as JsonSchemaValidationError
from pydantic import ValidationError as PydanticValidationError
from lmformatenforcer import JsonSchemaParser
from lmformatenforcer.integrations.transformers import build_transformers_prefix_allowed_tokens_fn

def make_case_runtime_state(case):
    history = []
    fixtures = case.get("fixtures", {})
    fixture = fixtures.get("persisted_result_fixture")
    if isinstance(fixture, dict):
        # The v4 checkpoint fixture is a trusted successful-operation record. Map its
        # persisted IDs into the app-shaped completed EDA result consumed by runtime.py.
        if fixture.get("tool") == "run_eda_analysis" and fixture.get("status") == "succeeded":
            data = {
                "id": fixture.get("analysis_id"),
                "dataset_id": fixture.get("dataset_id"),
                "file_id": fixture.get("file_id"),
                "status": "Completed",
            }
            app_result = {
                "tool": "run_eda_analysis",
                "status": "succeeded",
                "summary": fixture.get("summary", "Synthetic persisted EDA result."),
                "data": data,
                "references": [],
            }
        else:
            app_result = {
                key: fixture[key]
                for key in ("tool", "status", "summary", "data", "warnings", "references", "limitations")
                if key in fixture
            }
        history.append({"role": "assistant", "metadata": {"structured_results": [app_result]}})

    # Use only the synthetic verified profile state and evidence references. Do not copy
    # the fixture's gold objective, agent, runtime IDs, permissions, or contract context.
    specialist_fixture = fixtures.get("specialist_input_context")
    if isinstance(specialist_fixture, dict):
        profile_data = {
            "dataset_id": specialist_fixture.get("dataset_id"),
            "file_id": specialist_fixture.get("file_id"),
            "tables": [],
        }
        evidence_references = [
            {
                key: reference[key]
                for key in ("dataset_id", "file_id", "source_fingerprint", "source_sha256", "type", "kind", "resolution", "verification_status")
                if key in reference
            }
            for reference in specialist_fixture.get("evidence_references", [])
            if isinstance(reference, dict)
            and reference.get("type") == "verified_profile"
            and reference.get("dataset_id") == profile_data["dataset_id"]
            and reference.get("file_id") == profile_data["file_id"]
        ]
        if evidence_references:
            profile_result = {
                "tool": "get_profile",
                "status": "succeeded",
                "summary": "Synthetic verified profile fixture.",
                "data": profile_data,
                "references": evidence_references,
            }
            history.append({"role": "assistant", "metadata": {"structured_results": [profile_result]}})

    tool_result_fixture = case.get("tool_result_fixture")
    if (
        isinstance(tool_result_fixture, dict)
        and tool_result_fixture.get("status") == "succeeded"
        and isinstance(tool_result_fixture.get("data"), dict)
    ):
        history.append({"role": "assistant", "metadata": {"structured_results": [{
            "tool": tool_result_fixture.get("tool", "synthetic_tool_result"),
            "status": "succeeded",
            "summary": tool_result_fixture.get("summary", "Synthetic successful tool result."),
            "data": tool_result_fixture["data"],
            "references": [],
        }]}})

    state = runtime_helpers["_trusted_runtime_state"](history, [])
    return state, history


def verify_runtime_fixture_adapters():
    checkpoint_cases = [
        case for case in cases
        if isinstance(case.get("fixtures", {}).get("persisted_result_fixture"), dict)
        and case["fixtures"]["persisted_result_fixture"].get("tool") == "run_eda_analysis"
    ]
    for case in checkpoint_cases:
        state, _ = make_case_runtime_state(case)
        operations = state.get("prior_completed_engine_operations", [])
        fixture = case["fixtures"]["persisted_result_fixture"]
        if not any(
            operation.get("analysis_id") == fixture.get("analysis_id")
            and operation.get("dataset_id") == fixture.get("dataset_id")
            and operation.get("file_id") == fixture.get("file_id")
            for operation in operations
        ):
            raise RuntimeError("Persisted EDA fixture did not produce a trusted completed operation.")
    handoff_cases = [
        case for case in cases
        if isinstance(case.get("fixtures", {}).get("specialist_input_context"), dict)
    ]
    for case in handoff_cases:
        state, _ = make_case_runtime_state(case)
        if not state.get("prior_verified_profiles"):
            raise RuntimeError("Verified-profile fixture did not produce trusted profile state.")
    return {"completed_eda_fixture_cases": len(checkpoint_cases), "verified_profile_fixture_cases": len(handoff_cases)}

runtime_fixture_preflight = verify_runtime_fixture_adapters()
print("Runtime fixture preflight:", json.dumps(runtime_fixture_preflight, sort_keys=True))

def validate_tool_proposals(turn, case):
    allowed = {schema["function"]["name"]: schema for schema in case_tool_schemas(case)}
    outcomes = []
    for proposal in turn.tool_calls:
        item = {"name": proposal.name, "arguments": proposal.arguments, "accepted_for_preparation": False, "error": None}
        schema = allowed.get(proposal.name)
        if schema is None:
            item["error"] = "Tool is not registered in the case's available tool schemas."
        elif not isinstance(proposal.arguments, dict):
            item["error"] = "Tool arguments must be a JSON object."
        else:
            try:
                Draft202012Validator(schema["function"]["parameters"]).validate(proposal.arguments)
                item["accepted_for_preparation"] = True
            except JsonSchemaValidationError as error:
                item["error"] = {"path": list(error.absolute_path), "message": error.message}
        outcomes.append(item)
    return outcomes

def parse_and_validate(raw_text, case, runtime_state):
    parsed_json = json.loads(raw_text)
    turn = slim.SlimOrchestratorTurn.model_validate(parsed_json)
    state = runtime_state
    failed_counts, profiles, successful, completed_operations = runtime_helpers["_runtime_state_sets"](state)
    specialist_fixture = case.get("fixtures", {}).get("specialist_input_context")
    specialist_handoff_available = bool(
        isinstance(specialist_fixture, dict)
        and (specialist_fixture.get("dataset_id"), specialist_fixture.get("file_id")) in profiles
    )
    slim.validate_runtime_turn(
        turn,
        user_message=case["prompt"],
        trusted_failed_tool_counts=failed_counts,
        prior_verified_profiles=profiles,
        prior_successful_tools=successful,
        prior_completed_engine_operations=completed_operations,
        trusted_scope_available=bool(profiles),
        evidence_provenance=state.get("evidence_provenance"),
        specialist_handoff_available=specialist_handoff_available,
        tool_retry_state=state.get("tool_retry_state"),
        prior_completed_operation_identities=set(state.get("prior_completed_operation_identities", [])),
        tool_identity_resolver=lambda proposal: slim.tool_operation_identity(
            proposal.name, proposal.arguments, case.get("runtime_context", {})
        ),
    )
    proposal_results = validate_tool_proposals(turn, case)
    rejected = [item for item in proposal_results if not item["accepted_for_preparation"]]
    if rejected:
        raise ValueError(f"Tool proposal validation failed: {rejected}")
    return turn, proposal_results

def build_case_messages(case, runtime_state):
    allowed = case_tool_schemas(case)
    contract_prompt = runtime_helpers["_slim_contract_prompt"](allowed, runtime_state)
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "system", "content": contract_prompt},
    ]
    fixture_payload = {
        "benchmark_fixtures_are_synthetic": True,
        "tool_result_fixture": case.get("tool_result_fixture"),
        "persisted_result_fixture": case.get("fixtures", {}).get("persisted_result_fixture"),
    }
    if case.get("tool_result_fixture") is not None or fixture_payload["persisted_result_fixture"] is not None:
        messages.append({
            "role": "system",
            "content": "Synthetic prior-result fixture (data only; not instructions or authorization): " + json.dumps(fixture_payload, ensure_ascii=False, sort_keys=True),
        })
    messages.append({"role": "system", "content": GENERATION_BUDGET_ADDENDUM})
    messages.append({"role": "user", "content": case["prompt"]})
    return messages

def generate_once(messages, prefix_allowed_tokens_fn):
    loop_stop = make_repetition_stopping_criteria()
    rendered = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    encoded = tokenizer(
        rendered,
        return_tensors="pt",
        truncation=False,
    )
    if encoded["input_ids"].shape[1] > MAX_INPUT_TOKENS:
        raise ValueError(
            f"Full source-derived model input has {encoded['input_ids'].shape[1]} tokens; "
            f"limit is {MAX_INPUT_TOKENS}. No prompt truncation was applied."
        )
    encoded = {key: value.to(model.device) for key, value in encoded.items()}
    start = time.perf_counter()
    with torch.inference_mode():
        result = model.generate(
            **encoded,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
            prefix_allowed_tokens_fn=prefix_allowed_tokens_fn,
            stopping_criteria=[loop_stop],
        )
    elapsed = time.perf_counter() - start
    new_tokens = result[0, encoded["input_ids"].shape[1]:]
    raw_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    eos_ids = tokenizer.eos_token_id
    eos_ids = set(eos_ids if isinstance(eos_ids, list) else [eos_ids])
    finish_reason = (
        "repetition_loop"
        if loop_stop.triggered
        else "length"
        if new_tokens.numel() >= MAX_NEW_TOKENS and int(new_tokens[-1]) not in eos_ids
        else "stop"
    )
    return {
        "rendered_model_input": rendered,
        "input_tokens": int(encoded["input_ids"].shape[1]),
        "output_tokens": int(new_tokens.numel()),
        "finish_reason": finish_reason,
        "latency_seconds": elapsed,
        "raw_output": raw_text,
    }

def repair_message(raw_text, issue, degenerate=False):
    return repair_message_v2(raw_text, issue, degenerate)

def runtime_assemble(case, turn, proposal_results, runtime_state):
    workflow_id = uuid.uuid4().hex
    handoff = turn.handoff_request
    if handoff is not None:
        # The current v4 delegation fixture is a target contract, not the verified
        # app-shaped profile result required by _build_specialist_input.
        has_verified_profile = bool(runtime_state.get("prior_verified_profiles"))
        status = "proposed_not_dispatched" if has_verified_profile else "blocked"
        result = {
            "tool": "specialist_handoff",
            "status": status,
            "execution_status": "not_dispatched",
            "agent": handoff.agent,
            "objective": handoff.objective,
            "workflow_id": workflow_id,
            "task_id": uuid.uuid4().hex if has_verified_profile else None,
            "runtime_owned": True,
        }
        final_text = (
            "The specialist handoff was prepared but not dispatched."
            if has_verified_profile
            else "The specialist handoff was blocked because no verified profile result was available to the runtime."
        )
    elif turn.tool_calls:
        result = {
            "tool": "tool_proposals",
            "status": "prepared_not_dispatched",
            "execution_status": "not_dispatched",
            "proposals": [item for item in proposal_results],
            "workflow_id": workflow_id,
            "runtime_owned": True,
        }
        final_text = turn.response
    else:
        result = {
            "tool": None,
            "status": "no_dispatch",
            "execution_status": "not_dispatched",
            "workflow_id": workflow_id,
            "runtime_owned": True,
        }
        final_text = turn.response
    output = {
        "simulation": True,
        "case_id": case["id"],
        "expected": case,
        "execution_status": "simulated",
        "decision": turn.decision,
        "assistant_text": final_text,
        "model_response": turn.model_dump(mode="json"),
        "structured_result": result,
        "runtime_owned_field_integrity": (
            len(workflow_id) == 32
            and all(key not in turn.model_fields_set for key in ("workflow_id", "task_id", "evidence_references", "permitted_tools"))
        ),
    }
    structured = output["structured_result"]
    # Priority 4: the length policy applies to the FINAL assembled text, not only model output.
    length_issue = slim.final_response_length_issue(final_text or "")
    output["final_response_length_issue"] = length_issue
    workflow_id = structured.get("workflow_id")
    output["runtime_assembly_valid"] = (
        output["simulation"] is True
        and output["execution_status"] == "simulated"
        and output["case_id"] == case["id"]
        and structured.get("execution_status") == "not_dispatched"
        and structured.get("status") in {
            "proposed_not_dispatched",
            "blocked",
            "prepared_not_dispatched",
            "no_dispatch",
        }
        and (workflow_id is None or bool(re.fullmatch(r"[0-9a-f]{32}", workflow_id)))
        and output["runtime_owned_field_integrity"]
        and length_issue is None
    )
    return output

def generate_case(case):
    runtime_state, history = make_case_runtime_state(case)
    messages = build_case_messages(case, runtime_state)
    source_response_schema = runtime_helpers["response_json_schema"](case_tool_schemas(case))
    response_schema = tighten_generation_schema(source_response_schema)
    response_schema_digest = sha256_bytes(canonical(response_schema))
    source_response_schema_digest = sha256_bytes(canonical(source_response_schema))
    parser = JsonSchemaParser(response_schema)
    prefix_allowed_tokens_fn = build_transformers_prefix_allowed_tokens_fn(
        TOKENIZER_ENFORCER_DATA, parser
    )
    attempts = []
    accepted_turn = None
    accepted_proposals = []
    validation_error = None
    for attempt_number in range(3):
        try:
            generation = generate_once(messages, prefix_allowed_tokens_fn)
        except Exception as error:
            validation_error = f"GenerationError: {type(error).__name__}: {error}"
            attempts.append({
                "attempt": attempt_number + 1,
                "validation": "generation_error",
                "validation_error": validation_error,
                "raw_output": None,
                "latency_seconds": 0.0,
                "input_tokens": 0,
                "output_tokens": 0,
            })
            break
        attempt = {"attempt": attempt_number + 1, **generation, "validation": "failed", "validation_error": None}
        attempts.append(attempt)
        try:
            if generation["finish_reason"] in ("length", "repetition_loop"):
                raise ValueError(f"The response did not terminate ({generation['finish_reason']}).")
            accepted_turn, accepted_proposals = parse_and_validate(generation["raw_output"], case, runtime_state)
            attempt["tool_proposal_validation"] = accepted_proposals
            attempt["validation"] = "passed"
            break
        except (json.JSONDecodeError, PydanticValidationError, JsonSchemaValidationError, TypeError, ValueError, LookupError, PermissionError) as error:
            validation_error = f"{type(error).__name__}: {error}"
            attempt["validation_error"] = validation_error
            try:
                provisional = slim.SlimOrchestratorTurn.model_validate(json.loads(generation["raw_output"]))
                attempt["tool_proposal_validation"] = validate_tool_proposals(provisional, case)
            except (json.JSONDecodeError, PydanticValidationError, TypeError, ValueError):
                attempt["tool_proposal_validation"] = []
            if attempt_number == 2:
                break
            degenerate = generation["finish_reason"] in ("length", "repetition_loop")
            messages = messages + repair_message(generation["raw_output"], validation_error, degenerate)
    result = {
        "case_id": case["id"],
        "prompt": case["prompt"],
        "response_schema_sha256": response_schema_digest,
        "source_response_schema_sha256": source_response_schema_digest,
        "model_input_messages": messages,
        "trusted_runtime_state": runtime_state,
        "raw_outputs": attempts,
        "repair_attempts": max(0, len(attempts) - 1),
        "accepted": accepted_turn is not None,
        "validation_error": None if accepted_turn is not None else validation_error,
    }
    if accepted_turn is not None:
        result["parsed_response"] = accepted_turn.model_dump(mode="json")
        result["tool_proposal_validation"] = accepted_proposals
        result["runtime_assembled_response"] = runtime_assemble(case, accepted_turn, accepted_proposals, runtime_state)
    else:
        result["parsed_response"] = None
        result["tool_proposal_validation"] = (
            attempts[-1].get("tool_proposal_validation", [])
            if attempts
            else []
        )
        result["runtime_assembled_response"] = {
            "simulation": True,
            "case_id": case["id"],
            "execution_status": "safe_failure",
            "assistant_text": "I couldn't produce a response that passed runtime validation. No tool or specialist was dispatched. Please retry.",
            "structured_result": {"status": "failed", "execution_status": "not_dispatched"},
            "runtime_assembly_valid": True,
            "runtime_owned_field_integrity": True,
        }
        safe_output = result["runtime_assembled_response"]
        safe_result = safe_output["structured_result"]
        safe_output["runtime_assembly_valid"] = (
            safe_output["simulation"] is True
            and safe_output["execution_status"] == "safe_failure"
            and safe_output["case_id"] == case["id"]
            and safe_result.get("status") == "failed"
            and safe_result.get("execution_status") == "not_dispatched"
            and bool(safe_output.get("assistant_text"))
            and slim.final_response_length_issue(safe_output["assistant_text"]) is None
            and safe_output["runtime_owned_field_integrity"]
        )
    result["tool_proposal_validation_attempts"] = [
        {"attempt": item["attempt"], "results": item.get("tool_proposal_validation", [])}
        for item in attempts
        if item.get("tool_proposal_validation")
    ]
    return result

print("Adapter ready. Tool dispatch is disabled for every case.")

## 6. Run all 12 cases independently

Each case starts with fresh messages and fixture-derived trusted state. Only prior-result fixtures are passed as synthetic data; expected specialist-input contracts remain evaluator-only and are never exposed to the model or promoted to trusted state. Cases continue after a captured inference/validation failure, and no failed output is dropped.

In [ ]:
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
case_runs = []
for index, case in enumerate(cases, start=1):
    print(f"[{index:02d}/{len(cases)}] {case['id']}")
    try:
        case_runs.append(generate_case(case))
    except Exception as error:
        # Preserve unexpected case errors explicitly; continue only to retain other cases.
        case_runs.append({
            "case_id": case["id"],
            "prompt": case["prompt"],
            "model_input_messages": [],
            "trusted_runtime_state": {},
            "raw_outputs": [],
            "repair_attempts": 0,
            "accepted": False,
            "validation_error": f"Unhandled {type(error).__name__}: {error}",
            "parsed_response": None,
            "tool_proposal_validation": [],
            "runtime_assembled_response": {
                "simulation": True,
                "case_id": case["id"],
                "execution_status": "safe_failure",
                "assistant_text": "Case execution failed; inspect the captured error.",
                "structured_result": {"status": "failed", "execution_status": "not_dispatched"},
                "runtime_assembly_valid": True,
                "runtime_owned_field_integrity": True,
            },
        })
    if torch.cuda.is_available():
        torch.cuda.synchronize()

if len(case_runs) != 12 or {item["case_id"] for item in case_runs} != set(case_ids):
    raise RuntimeError("Run did not preserve exactly one result per canonical benchmark case.")
print(f"Captured {len(case_runs)} independent cases for run {run_id}.")

## 7. Bounded repair and runtime regression probes

The source validator and runtime helpers are exercised against explicit probes. These are adapter tests, not extra benchmark cases or model outputs. They cover operation-specific retry eligibility, persisted attempts, clarification continuation, checkpoint replay protection, final-response shortening, and handoff non-dispatch.

In [ ]:
from pydantic import ValidationError as PydanticValidationError

def valid_turn(**overrides):
    value = {
        "decision": "answer", "intent": "Answer directly.", "plan": [],
        "tool_calls": [], "handoff_request": None, "response": "A response.",
        "clarification_questions": [],
    }
    value.update(overrides)
    return slim.SlimOrchestratorTurn.model_validate(value)

def policy_call(turn, prompt, failed=None, profiles=None, successful=None):
    return slim.validate_runtime_turn(
        turn, user_message=prompt,
        trusted_failed_tool_counts=failed or {},
        prior_verified_profiles=profiles or set(),
        prior_successful_tools=successful or set(),
    )

def outcome(name, fn, should_pass):
    try:
        fn()
        actual = True
        detail = "accepted"
    except Exception as error:
        actual = False
        detail = f"{type(error).__name__}: {error}"
    passed = actual == should_pass
    return {"probe": name, "passed": passed, "expected_accept": should_pass, "detail": detail}

policy_probes = [
    outcome("quoted numeric identifier requires clarification", lambda: policy_call(
        valid_turn(decision="clarify", clarification_questions=["Please provide the dataset ID as an integer."]),
        'Get dataset "42".',
    ), True),
    outcome("quoted identifier cannot be coerced into a tool call", lambda: policy_call(
        valid_turn(decision="continue", tool_calls=[{"name":"get_profile","arguments":{"dataset_id":42,"file_id":7}}]),
        'Get profile for dataset "42" and file 7.',
    ), False),
    outcome("causal assertion request must abstain", lambda: policy_call(
        valid_turn(decision="abstain", response="Timing or correlation does not establish causation. A comparison group is required to support a causal conclusion."),
        "Tell me what caused the change.",
    ), True),
    outcome("execution claim without trusted tool success is rejected", lambda: policy_call(
        valid_turn(response="I ran the analysis."),
        "What happened?",
    ), False),
    outcome("repeat verified profile lookup is rejected", lambda: policy_call(
        valid_turn(decision="continue", tool_calls=[{"name":"get_profile","arguments":{"dataset_id":42,"file_id":7}}]),
        "Get the profile.",
        profiles={(42, 7)},
    ), False),
    outcome("dependent profile and quality calls in one turn are rejected", lambda: policy_call(
        valid_turn(decision="continue", tool_calls=[
            {"name":"get_profile","arguments":{"dataset_id":42,"file_id":7}},
            {"name":"run_quality_analysis","arguments":{"dataset_id":42,"file_id":7}},
        ]),
        "Inspect then run quality.",
    ), False),
    outcome("same-tool proposal after trusted failure is rejected", lambda: policy_call(
        valid_turn(decision="continue", tool_calls=[{"name":"list_files","arguments":{"dataset_id":42}}]),
        "List files.",
        failed={"list_files":1},
    ), False),
    outcome("runtime-owned response fields are rejected by slim DTO", lambda: slim.SlimOrchestratorTurn.model_validate({
        **valid_turn().model_dump(mode="json"), "workflow_id":"a"*32,
    }), False),
    outcome("tool schema rejects string ID", lambda: Draft202012Validator(
        tool_schema_by_name["get_profile"]["function"]["parameters"]
    ).validate({"dataset_id":"42","file_id":7}), False),
]
if not all(item["passed"] for item in policy_probes):
    raise RuntimeError(f"One or more source policy probes failed: {policy_probes}")
policy_probe_boundaries = {
    "handoff_without_verified_profile": "runtime blocks it; an explicit delegation request requires a matching trusted profile, and the specialist is not dispatched",
    "checkpoint_resume": "only a matching trusted completed EDA checkpoint can authorize historical wording; resume proposes get_eda_result and forbids engine replay",
    "engine_prerequisites": "Kaggle does not execute _engine_intent_allowed or engine operations; tool dispatch remains disabled",
    "prior_failure_acknowledgement": "source validator rejects repeating the failed tool; semantic acknowledgement remains prompt/human-review behavior",
    "unsupported evidence references": "the slim DTO rejects top-level evidence_references; evidence grounding is evaluated separately and heuristically",
    "dependent calls": "source validator explicitly rejects same-turn get_profile + run_quality_analysis",
}
print(json.dumps({"source_policy_probes": policy_probes, "simulation_boundaries": policy_probe_boundaries}, indent=2))

def validate_retry_probe(retry_state, retry_args, retry_context):
    turn = valid_turn(
        decision="continue",
        tool_calls=[{"name": "list_files", "arguments": retry_args}],
        response="I will list the files.",
    )
    return slim.validate_runtime_turn(
        turn,
        user_message="List files for the selected dataset.",
        trusted_failed_tool_counts={"list_files": 1},
        prior_verified_profiles=set(),
        prior_successful_tools=set(),
        trusted_scope_available=True,
        tool_retry_state=retry_state,
        tool_identity_resolver=lambda proposal: slim.tool_operation_identity(
            proposal.name, proposal.arguments, retry_context
        ),
    )

retry_args = {"dataset_id": 42}
retry_context = {"dataset_id": 42}
retry_identity = slim.tool_operation_identity("list_files", retry_args, retry_context)
retry_failure = {
    "tool": "list_files", "status": "failed", "retryable": True,
    "scheduler_attempts": 1, "scheduler_max_retries": 1,
    "scheduler_task_id": "tool:runtime-probe",
}
retry_state = slim.build_tool_retry_state({retry_identity: [retry_failure]})
unknown_retry_state = slim.build_tool_retry_state({retry_identity: [{
    **retry_failure, "retryable": None, "scheduler_attempts": None,
    "scheduler_max_retries": None,
}]})

def retry_state_probe():
    statuses = {
        key: slim.build_tool_retry_state({retry_identity: [{
            **retry_failure, "retryable": retryable,
            "scheduler_attempts": attempts, "scheduler_max_retries": 1,
        }]})[retry_identity]["status"]
        for key, retryable, attempts in (
            ("permitted", True, 1), ("exhausted", True, 2),
            ("not_retryable", False, 1), ("unknown", None, None),
        )
    }
    if statuses != {
        "permitted": "retry_permitted", "exhausted": "retry_exhausted",
        "not_retryable": "not_retryable", "unknown": "unknown",
    }:
        raise AssertionError(f"Unexpected persisted retry states: {statuses}")

runtime_probe_context = {
    "workflow_id": "2" * 32,
    "pending_clarification": {
        "workflow_id": "1" * 32,
        "questions": ["Which reporting period should I use?"],
    },
}
protected_workflows = runtime_helpers["_protected_workflow_ids"](runtime_probe_context)
clarification_args = {"dataset_id": 42, "file_id": 7}
clarification_identity = slim.tool_operation_identity(
    "run_quality_analysis", clarification_args, {"dataset_id": 42}
)
clarification_result = {
    "tool": "run_quality_analysis", "status": "succeeded",
    "retry_identity": clarification_identity,
    "scheduler_workflow_id": "1" * 32,
}
clarification_state = runtime_helpers["_trusted_runtime_state"](
    [{"role": "assistant", "metadata": {"structured_results": [clarification_result]}}],
    [], protected_workflow_ids=protected_workflows,
)

def clarification_probe():
    pending = runtime_helpers["_pending_clarification_from_context"](runtime_probe_context)
    if pending is None or clarification_identity not in clarification_state["prior_completed_operation_identities"]:
        raise AssertionError("Clarification continuation lost its originating completed operation.")

checkpoint_args = {
    "dataset_id": 42, "file_id": 7, "analysis_type": "univariate",
    "selected_columns": ["sales"],
}
checkpoint_context = {"dataset_id": 42, "file_id": 7}
checkpoint_identity = slim.tool_operation_identity(
    "run_eda_analysis", checkpoint_args, checkpoint_context
)
checkpoint_turn = valid_turn(
    decision="continue",
    tool_calls=[{"name": "run_eda_analysis", "arguments": checkpoint_args}],
    response="I will run the requested EDA.",
)

def reject_checkpoint_replay():
    return slim.validate_runtime_turn(
        checkpoint_turn,
        user_message="Run EDA analysis for the selected file.",
        trusted_failed_tool_counts={},
        prior_verified_profiles={(42, 7)},
        prior_successful_tools=set(),
        trusted_scope_available=True,
        prior_completed_operation_identities={checkpoint_identity},
        tool_identity_resolver=lambda proposal: slim.tool_operation_identity(
            proposal.name, proposal.arguments, checkpoint_context
        ),
    )

long_outcome = contracts.AgentOutcome(
    status="completed",
    assistant_text=" ".join(f"Result {index} is available." for index in range(40)),
    activities=(), structured_results=(), warnings=(), usage=contracts.UsageMetadata(),
)
corrected_outcome = runtime_helpers["_enforce_final_response_policy"](long_outcome)

def final_response_probe():
    if corrected_outcome.status != "partial" or slim.final_response_length_issue(corrected_outcome.assistant_text) is not None:
        raise AssertionError("The final assembled response was not shortened to source policy limits.")

handoff_turn = valid_turn(
    decision="continue",
    handoff_request={"agent": "data_intelligence", "objective": "Review the verified profile."},
    response="I will request specialist review.",
)
handoff_output = runtime_assemble(
    {"id": "runtime-handoff-probe"}, handoff_turn, [],
    {"prior_verified_profiles": [{"dataset_id": 42, "file_id": 7}]},
)

def handoff_non_dispatch_probe():
    result = handoff_output["structured_result"]
    if result.get("status") != "proposed_not_dispatched" or result.get("execution_status") != "not_dispatched":
        raise AssertionError("The simulated specialist handoff crossed the dispatch boundary.")

def distinct_operation_identity_probe():
    changed_identity = slim.tool_operation_identity(
        "list_files", {"dataset_id": 43}, {"dataset_id": 43}
    )
    if changed_identity == retry_identity:
        raise AssertionError("Distinct operation inputs shared an identity.")

runtime_simulation_probes = [
    outcome("runtime: persisted retry-state distinctions", retry_state_probe, True),
    outcome("runtime: matching operation retry is permitted", lambda: validate_retry_probe(retry_state, retry_args, retry_context), True),
    outcome("runtime: unknown retry eligibility is rejected", lambda: validate_retry_probe(unknown_retry_state, retry_args, retry_context), False),
    outcome("runtime: changed arguments produce a distinct operation identity", lambda: distinct_operation_identity_probe(), True),
    outcome("runtime: clarification preserves originating operation protection", clarification_probe, True),
    outcome("runtime: completed checkpoint prevents engine replay", reject_checkpoint_replay, False),
    outcome("runtime: final response enforcement shortens assembled output", final_response_probe, True),
    outcome("runtime: specialist handoff remains undispatched", handoff_non_dispatch_probe, True),
]
policy_probes.extend(runtime_simulation_probes)
if not all(item["passed"] for item in runtime_simulation_probes):
    raise RuntimeError(f"One or more runtime simulation probes failed: {runtime_simulation_probes}")
print("Runtime simulation probes:", json.dumps(runtime_simulation_probes, indent=2))


## 8. Scoring and runtime-simulation results

Decision accuracy is counted only when the complete slim response is accepted. Exact tool-sequence scoring includes every case that explicitly specifies `tool_calls_exact`, including `[]`. Every such case remains in the denominator after invalid model output or repair exhaustion. Human semantic review is still required; string-pattern safety checks are heuristics, not proof of grounding.

In [ ]:
EXECUTION_CLAIM = re.compile(
    r"\b(?:i|we)\s+(?:have\s+)?(?:run|ran|executed|performed|built|created|completed)\b"
    r"|\b(?:the analysis|dashboard|report)\s+(?:has been|was)\s+(?:run|executed|created|built)\b"
    r"|\b(?:profile|dataset|file|report|analysis|dashboard)\b[^.!?\n]{0,40}\b(?:retrieved|loaded|inspected|checked|completed|created)\b",
    re.IGNORECASE,
)
# CAUSAL_CLAIM / unsupported_causal_claim() come from section 3b (negation-aware).
REFERENCE_PATTERN = re.compile(r"\b(?:dataset|file)(?:[\s_]+id)?[\"']?\s*[#:=]?\s*[\"']?(\d+)\b", re.IGNORECASE)
RETRIEVAL_CLAIM = re.compile(
    r"\b(?:i|we)\s+(?:have\s+)?(?:retrieved|loaded|inspected|checked)\b"
    r"|\b(?:profile|dataset|file|result)\s+(?:was\s+)?(?:retrieved|loaded|inspected|checked)\b",
    re.IGNORECASE,
)
NUMBER_PATTERN = re.compile(r"(?<![A-Za-z])\d+(?:\.\d+)?%?")
POLICY_ERROR_MARKERS = (
    "quoted string identifiers",
    "unsupported causal assertions",
    "execution claims require",
    "verified profile lookup",
    "quality analysis depends",
    "prior ",
)

def ratio(numerator, denominator):
    return {"numerator": int(numerator), "denominator": int(denominator), "rate": (numerator / denominator if denominator else None)}

case_by_id = {case["id"]: case for case in cases}
rows = []
for run in case_runs:
    case = case_by_id[run["case_id"]]
    turn = run.get("parsed_response")
    runtime_output = run.get("runtime_assembled_response") or {}
    expected = case.get("expected", {})
    text = ""
    if turn:
        text = " ".join([turn.get("intent", ""), *turn.get("plan", []), turn.get("response", "")])
    trusted_success = bool(run.get("trusted_runtime_state", {}).get("prior_successful_tools"))
    exact_tools_expected = "tool_calls_exact" in expected
    actual_calls = turn.get("tool_calls", []) if turn else []
    out_refs = [int(value) for value in REFERENCE_PATTERN.findall(text)]
    evidence_text = json.dumps({
        "prompt": case["prompt"],
        "fixtures": case.get("fixtures", {}),
        "tool_result_fixture": case.get("tool_result_fixture"),
    }, ensure_ascii=False, sort_keys=True)
    in_refs = [int(value) for value in REFERENCE_PATTERN.findall(evidence_text)]
    known_numbers = set(NUMBER_PATTERN.findall(evidence_text))
    unsupported_numbers = set(NUMBER_PATTERN.findall(text)) - known_numbers
    proposal_refs = [
        value
        for call in actual_calls
        if isinstance(call, dict) and isinstance(call.get("arguments"), dict)
        for key in ("dataset_id", "file_id")
        if isinstance((value := call["arguments"].get(key)), int) and not isinstance(value, bool)
    ]
    exact_tool_match = bool(
        turn is not None
        and actual_calls == expected.get("tool_calls_exact")
    ) if exact_tools_expected else None
    expected_calls = expected.get("tool_calls_exact", [])
    tool_selection_match = bool(
        turn is not None
        and [call.get("name") for call in actual_calls] == [call.get("name") for call in expected_calls]
    ) if exact_tools_expected else None
    tool_arguments_match = bool(
        turn is not None
        and len(actual_calls) == len(expected_calls)
        and all(
            actual.get("arguments") == gold_call.get("arguments")
            for actual, gold_call in zip(actual_calls, expected_calls)
        )
    ) if exact_tools_expected else None
    clarification_expected = expected.get(
        "clarification_required",
        expected.get("decision") == "clarify",
    )
    actual_clarification = bool(turn and turn.get("decision") == "clarify" and turn.get("clarification_questions"))
    handoff_expected = isinstance(case.get("fixtures", {}).get("specialist_input_context"), dict)
    handoff_actual = bool(turn and turn.get("handoff_request"))
    schema_valid = bool(run.get("accepted"))
    row = {
        "case_id": case["id"],
        "prompt": case["prompt"],
        "expected": case,
        "expected_behavior": case.get("expected_behavior", expected.get("reason", "")),
        "expected_decision": expected.get("decision"),
        "actual_decision": turn.get("decision") if turn else None,
        "schema_valid": schema_valid,
        "decision_correct": bool(schema_valid and expected.get("decision") == turn.get("decision")),
        "tool_sequence_expected": exact_tools_expected,
        "expected_tool_calls": expected.get("tool_calls_exact"),
        "actual_tool_calls": actual_calls if turn else None,
        "tool_selection_correct": tool_selection_match,
        "tool_arguments_correct": tool_arguments_match,
        "tool_sequence_correct": exact_tool_match,
        "tool_proposals_valid": bool(schema_valid and all(item.get("accepted_for_preparation") for item in run.get("tool_proposal_validation", []))),
        "clarification_expected": bool(clarification_expected),
        "clarification_correct": bool(schema_valid and bool(clarification_expected) == actual_clarification),
        "handoff_expected": handoff_expected,
        "handoff_actual": handoff_actual,
        "handoff_missing": bool(handoff_expected and not handoff_actual),
        "handoff_unnecessary": bool(not handoff_expected and handoff_actual),
        "model_output": turn.get("response") if turn else None,
        "runtime_output": runtime_output.get("assistant_text"),
        "runtime_status": runtime_output.get("structured_result", {}).get("status"),
        "runtime_assembly_valid": bool(runtime_output.get("runtime_assembly_valid")),
        "final_response_length_valid": slim.final_response_length_issue(runtime_output.get("assistant_text") or "") is None,
        "runtime_owned_fields_valid": bool(runtime_output.get("runtime_owned_field_integrity")),
        "repair_attempts": run.get("repair_attempts", 0),
        "latency_seconds": round(sum(item.get("latency_seconds", 0) for item in run.get("raw_outputs", [])), 4),
        "input_tokens": sum(item.get("input_tokens", 0) for item in run.get("raw_outputs", [])),
        "output_tokens": sum(item.get("output_tokens", 0) for item in run.get("raw_outputs", [])),
        "unsupported_execution_claim_heuristic": bool(EXECUTION_CLAIM.search(text)) and not trusted_success,
        "unsupported_retrieval_claim_heuristic": bool(RETRIEVAL_CLAIM.search(text)) and not trusted_success,
        "unsupported_causal_claim_heuristic": unsupported_causal_claim(text),
        "unsupported_reference_heuristic": any(value not in in_refs for value in [*out_refs, *proposal_refs]),
        "unsupported_numeric_claim_heuristic": bool(unsupported_numbers),
        "policy_rejected_attempt": any(
            marker in str(attempt.get("validation_error", "")).casefold()
            for attempt in run.get("raw_outputs", [])
            for marker in POLICY_ERROR_MARKERS
        ),
        "review_required": True,
        "reviewer_decision": "",
        "review_notes": "",
        "error": run.get("validation_error"),
    }
    rows.append(row)

def aggregate_metric(key, predicate=None):
    selected = [row for row in rows if predicate is None or predicate(row)]
    return ratio(sum(bool(row.get(key)) for row in selected), len(selected))

tool_sequence_rows = [row for row in rows if row["tool_sequence_expected"]]
clarification_rows = rows
final_proposals = [item for run in case_runs for item in run.get("tool_proposal_validation", [])]
attempt_proposals = [
    item
    for run in case_runs
    for attempt in run.get("tool_proposal_validation_attempts", [])
    for item in attempt["results"]
]
metrics = {
    "case_count": len(rows),
    "model": {
        "slim_schema_validity": ratio(sum(row["schema_valid"] for row in rows), len(rows)),
        "decision_accuracy": ratio(sum(row["decision_correct"] for row in rows), len(rows)),
        "tool_selection_accuracy": ratio(sum(row["tool_selection_correct"] is True for row in tool_sequence_rows), len(tool_sequence_rows)),
        "tool_arguments_expected_accuracy": ratio(sum(row["tool_arguments_correct"] is True for row in tool_sequence_rows), len(tool_sequence_rows)),
        "exact_tool_sequence_accuracy_including_empty": ratio(sum(row["tool_sequence_correct"] is True for row in tool_sequence_rows), len(tool_sequence_rows)),
        "tool_proposal_validity_final_response": ratio(
            sum(item["accepted_for_preparation"] for item in final_proposals),
            len(final_proposals),
        ),
        "tool_proposal_validity_all_generated_attempts": ratio(
            sum(item["accepted_for_preparation"] for item in attempt_proposals),
            len(attempt_proposals),
        ),
        "clarification_accuracy": ratio(sum(row["clarification_correct"] for row in clarification_rows), len(clarification_rows)),
        "specialist_handoff_presence_accuracy": ratio(sum(row["handoff_actual"] == row["handoff_expected"] for row in rows), len(rows)),
        "missing_required_handoffs": sum(row["handoff_missing"] for row in rows),
        "unnecessary_handoffs": sum(row["handoff_unnecessary"] for row in rows),
        "unsupported_execution_claim_heuristic_rate": ratio(sum(row["unsupported_execution_claim_heuristic"] for row in rows), len(rows)),
        "unsupported_retrieval_claim_heuristic_rate": ratio(sum(row["unsupported_retrieval_claim_heuristic"] for row in rows), len(rows)),
        "unsupported_causal_claim_heuristic_rate": ratio(sum(row["unsupported_causal_claim_heuristic"] for row in rows), len(rows)),
        "unsupported_reference_heuristic_rate": ratio(sum(row["unsupported_reference_heuristic"] for row in rows), len(rows)),
        "unsupported_numeric_claim_heuristic_rate": ratio(sum(row["unsupported_numeric_claim_heuristic"] for row in rows), len(rows)),
        "source_policy_rejection_cases": ratio(sum(row["policy_rejected_attempt"] for row in rows), len(rows)),
        "policy_repair_exhaustion_rate": ratio(sum(not run.get("accepted") for run in case_runs), len(case_runs)),
    },
    "runtime_simulation": {
        "assembly_validity": ratio(sum(row["runtime_assembly_valid"] for row in rows), len(rows)),
        "runtime_owned_field_integrity": ratio(sum(row["runtime_owned_fields_valid"] for row in rows), len(rows)),
        "proposal_preparation_validation": ratio(
            sum(item["accepted_for_preparation"] for item in final_proposals),
            len(final_proposals),
        ),
        "proposal_validation_by_attempt": ratio(
            sum(item["accepted_for_preparation"] for item in attempt_proposals),
            len(attempt_proposals),
        ),
        "repairs_needed_cases": ratio(sum(row["repair_attempts"] > 0 for row in rows), len(rows)),
        "repair_success_rate": ratio(
            sum(run.get("accepted") and run.get("repair_attempts", 0) > 0 for run in case_runs),
            sum(run.get("repair_attempts", 0) > 0 for run in case_runs),
        ),
        "safe_failure_cases": sum(not run.get("accepted") for run in case_runs),
        "safe_failure_rate": ratio(
            sum(not run.get("accepted") and run.get("runtime_assembled_response", {}).get("execution_status") == "safe_failure" for run in case_runs),
            sum(not run.get("accepted") for run in case_runs),
        ),
        "final_response_correctness": {
            "numerator": None,
            "denominator": 0,
            "rate": None,
            "status": "human_semantic_review_pending",
        },
        "assembled_final_response_length_compliance": ratio(
            sum(row["final_response_length_valid"] for row in rows), len(rows)
        ),
        "runtime_regression_probe_pass_rate": ratio(
            sum(item["passed"] for item in runtime_simulation_probes),
            len(runtime_simulation_probes),
        ),
        "runtime_regression_probes": runtime_simulation_probes,
        "tool_dispatch_count": 0,
        "specialist_dispatch_count": 0,
    },
    "performance": {
        "latency_mean_seconds": statistics.mean(row["latency_seconds"] for row in rows) if rows else None,
        "latency_median_seconds": statistics.median(row["latency_seconds"] for row in rows) if rows else None,
        "latency_p95_seconds": sorted(row["latency_seconds"] for row in rows)[max(0, int(0.95 * len(rows) + 0.999999) - 1)] if rows else None,
        "input_tokens": sum(row["input_tokens"] for row in rows),
        "output_tokens": sum(row["output_tokens"] for row in rows),
        "generation_attempts": sum(len(run.get("raw_outputs", [])) for run in case_runs),
        "gpu_peak_allocated_gib": round(torch.cuda.max_memory_allocated() / (1024 ** 3), 3),
    },
    "comparison": {
        "original_benchmark": benchmark["benchmark_version"],
        "original_baseline_report_status": bundle.get("baseline_execution_status"),
        "original_baseline_metrics": None,
        "comparison_status": "not_comparable_no_verified_original_run_metrics",
        "reason": "The bundled v4 baseline report has status pending and metrics null; no raw original run is included.",
    },
}
if any(row["repair_attempts"] > 2 for row in rows):
    raise RuntimeError("Repair attempts exceeded the application limit.")
print(json.dumps(metrics, indent=2, allow_nan=False))

## 9. Human review

All 12 cases require semantic review. `reviewer_decision` and `review_notes` remain blank until a human completes the CSV. Automated outputs are not human approvals.

In [ ]:
human_review = [
    {
        "case_id": row["case_id"],
        "prompt": row["prompt"],
        "expected_behavior": row["expected_behavior"],
        "model_output": row["model_output"],
        "runtime_output": row["runtime_output"],
        "automated_result": json.dumps({
            "schema_valid": row["schema_valid"],
            "decision_correct": row["decision_correct"],
            "tool_sequence_correct": row["tool_sequence_correct"],
            "runtime_status": row["runtime_status"],
            "error": row["error"],
        }, ensure_ascii=False),
        "review_required": True,
        "reviewer_decision": "",
        "notes": "",
    }
    for row in rows
]
for review_row in human_review:
    print(json.dumps(review_row, ensure_ascii=False))
print(f"Human review pending for {len(human_review)}/{len(cases)} cases.")

## 10. Export, manifest and ZIP integrity

The export is unique per run and includes raw attempts, parsed proposals, simulated runtime outputs, per-case metrics, aggregate metrics, human-review CSV, configuration, and a Markdown summary. The manifest hashes every listed file. A final audit verifies paths, SHA-256 values and ZIP CRC before presenting the run directory.

In [ ]:
OUTPUT_ROOT = Path("/kaggle/working/insightos-runtime-evaluation")
EXPORT_RESERVE_GIB = 0.5
disk_before_export = shutil.disk_usage("/kaggle/working")
if disk_before_export.free < EXPORT_RESERVE_GIB * (1024 ** 3):
    raise RuntimeError(
        f"Only {disk_before_export.free / (1024 ** 3):.2f} GiB remains before export; "
        f"keep at least {EXPORT_RESERVE_GIB:.1f} GiB free. "
        "Inference may have completed, but export was stopped to protect the filesystem."
    )
RUN_DIR = OUTPUT_ROOT / run_id
RUN_DIR.mkdir(parents=True, exist_ok=False)

def write_json(name, value):
    path = RUN_DIR / name
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    return path

raw_outputs = {
    "run_id": run_id,
    "benchmark_version": benchmark["benchmark_version"],
    "cases": [{"case_id": run["case_id"], "prompt": run["prompt"], "expected": case_by_id[run["case_id"]], "attempts": run["raw_outputs"]} for run in case_runs],
}
parsed_proposals = {
    "run_id": run_id,
    "cases": [{"case_id": run["case_id"], "expected": case_by_id[run["case_id"]], "accepted": run["accepted"], "parsed_response": run["parsed_response"], "tool_proposal_validation": run["tool_proposal_validation"], "validation_error": run["validation_error"]} for run in case_runs],
}
runtime_outputs = {"run_id": run_id, "simulation": True, "cases": [run["runtime_assembled_response"] for run in case_runs], "regression_probes": runtime_simulation_probes}
case_results = {"run_id": run_id, "cases": rows}
configuration = {
    "run_id": run_id,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "notebook_version": "InsightOS_Qwen3_8B_Runtime_Evaluation_v2.5.0_RUNTIME_STATE_REGRESSION",
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "tokenizer_id": MODEL_ID,
    "tokenizer_revision": MODEL_REVISION,
    "benchmark_version": benchmark["benchmark_version"],
    "benchmark_sha256": EXPECTED_INPUT_HASHES["benchmark/qwen3-8b-orchestration-benchmark-v4.json"],
    "gold_response_sha256": EXPECTED_INPUT_HASHES["benchmark/qwen3-8b-gold-responses-v1.json"],
    "few_shot_sha256": EXPECTED_INPUT_HASHES["application/app/agent/orchestrator_few_shot_v1.json"],
    "input_hashes": EXPECTED_INPUT_HASHES,
    "environment": environment,
    "generation": GENERATION,
    "quantization": {"load_in_4bit": True, "quant_type": "nf4", "compute_dtype": "float16", "double_quant": True},
    "cuda_preflight": cuda_preflight,
    "model_load_seconds": model_load_seconds,
    "adapter": "notebook-local source-grounded simulation with JSON Schema constrained decoding; no production app/tool dispatch",
    "repair_attempt_limit": 2,
    "structured_output": {"method": "lm-format-enforcer", "schema_source": "current application response_json_schema, tightened with length bounds for decoding only", "prompt_contains_schema_object": False},
    "source_policy_changes": {"tool_retry_state": "retry_permitted | retry_exhausted | not_retryable with recovery_tools; one budget shared by scheduler and model", "final_response_length": "source-enforced abbreviation-aware 1-3 sentences and <=900 chars on final outcomes", "operation_retry": "persisted attempts and retryability are keyed by tool/input/scope; unknown eligibility never grants retry", "clarification": "protects originating workflow operations across continuation", "checkpoint": "completed engine operation identities cannot be replayed", "handoff": "proposal-only; specialist dispatch remains disabled", "bundle": "v2.5.0 (current runtime, scheduler, graph, checkpointer, and regression-test snapshots)"},
    "generation_hardening": {"caps": GENERATION_CAPS, "budget_addendum": GENERATION_BUDGET_ADDENDUM, "max_new_tokens": MAX_NEW_TOKENS, "repetition_stop": {"window": 16, "repeats": 3}, "repair_echoes_degenerate_output": False},
    "specialists_dispatched": False,
    "policy_probes": policy_probes,
    "policy_probe_boundaries": policy_probe_boundaries,
}
write_json("raw_model_outputs.json", raw_outputs)
write_json("parsed_model_proposals.json", parsed_proposals)
write_json("runtime_assembled_outputs.json", runtime_outputs)
write_json("per_case_evaluation_results.json", case_results)
write_json("aggregate_metrics.json", metrics)
write_json("execution_configuration.json", configuration)
with (RUN_DIR / "human_review.csv").open("w", encoding="utf-8", newline="") as output_file:
    writer = csv.DictWriter(output_file, fieldnames=list(human_review[0]), quoting=csv.QUOTE_MINIMAL)
    writer.writeheader()
    writer.writerows(human_review)

summary_lines = [
    f"# Runtime-Aware Evaluation Summary — {run_id}",
    "",
    f"- Benchmark: `{benchmark['benchmark_version']}` ({len(cases)} cases)",
    f"- Model: `{MODEL_ID}` at `{MODEL_REVISION}`",
    "- Quantization: 4-bit NF4, FP16 compute, double quantization",
    "- Decode: greedy, fixed seed, thinking disabled",
    "- Runtime mode: simulated; tools and specialists were not dispatched",
    "- Structured output: constrained by a length-bounded tightening of the source-generated response JSON Schema (outputs still satisfy the source schema); runtime policy validation and repairs remain active.",
    "- Fresh pinned-model evaluation: pending until this packaged notebook is executed on Kaggle.",
    "- Human review: pending for every case",
    "",
    "## Key metrics (numerator / denominator)",
    "",
]
for key, value in metrics["model"].items():
    if isinstance(value, dict) and "numerator" in value:
        summary_lines.append(f"- {key}: {value['numerator']} / {value['denominator']} ({value['rate']})")
    else:
        summary_lines.append(f"- {key}: {value}")
summary_lines += [
    "",
    "## Interpretation limitations",
    "",
    "- These results measure the Qwen3-8B model proposal plus a notebook-local runtime simulation, not production execution.",
    "- Tool operations and specialist handoffs are not run; generated proposals remain unexecuted.",
    "- Automated claim checks are heuristics. Review every row in `human_review.csv`.",
    "- The packaged original v4 baseline has no validated metrics; direct numerical comparison is unavailable.",
]
(RUN_DIR / "evaluation_summary.md").write_text("\n".join(summary_lines) + "\n", encoding="utf-8")

artifact_paths = sorted(path for path in RUN_DIR.iterdir() if path.is_file())
manifest_files = {path.name: sha256_bytes(path.read_bytes()) for path in artifact_paths}
manifest = {
    "manifest_version": 1,
    "run_id": run_id,
    "benchmark_version": benchmark["benchmark_version"],
    "files": manifest_files,
}
write_json("sha256_manifest.json", manifest)
for name, expected_hash in manifest_files.items():
    path = RUN_DIR / name
    if not path.is_file() or sha256_bytes(path.read_bytes()) != expected_hash:
        raise RuntimeError(f"Manifest verification failed for {name}")

zip_path = OUTPUT_ROOT / f"InsightOS_Qwen3_8B_Runtime_Evaluation_{run_id}.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN_DIR.iterdir()):
        if path.is_file():
            archive.write(path, arcname=f"{RUN_DIR.name}/{path.name}")
with zipfile.ZipFile(zip_path) as archive:
    bad_member = archive.testzip()
    if bad_member:
        raise RuntimeError(f"Evaluation ZIP CRC failed at {bad_member}")
    for name, expected_hash in manifest_files.items():
        if sha256_bytes(archive.read(f"{RUN_DIR.name}/{name}")) != expected_hash:
            raise RuntimeError(f"ZIP artifact hash mismatch for {name}")
print(json.dumps({"run_dir": str(RUN_DIR), "zip": str(zip_path), "manifest_files": len(manifest_files), "zip_integrity": "passed"}, indent=2))

## 11. Baseline comparison and interpretation

The bundled v4 reference report remains pending and is not substituted for validated run metrics. The previously supplied original baseline archive is preserved as an immutable reference; compare its case records only after independently validating its manifest and matching the scoring contract. This fresh run remains a notebook simulation, not production execution.
### Leakage-control declaration

This run is zero-shot with respect to the 12 benchmark cases. The bundled production few-shot file is retained only as a hashed input/audit artifact. Its overlapping examples are not supplied to the model. Do not compare this run to a production prompt run with few-shot demonstrations as though the prompt conditions were identical.


In [ ]:
comparison_status = metrics["comparison"]["comparison_status"]
assert comparison_status == "not_comparable_no_verified_original_run_metrics"
print(metrics["comparison"]["reason"])
print("Do not authorize fine-tuning from automated scores alone; review the 12-case table and source/runtime limitations first.")

## 12. Run completion checklist

- [ ] Notebook executed on Kaggle GPU at the pinned revision.
- [ ] All 12 case outputs and errors preserved.
- [ ] Aggregate denominators checked, including expected empty tool sequences.
- [ ] Human reviewer completed the exported CSV.
- [ ] Runtime simulation limitations acknowledged.
- [ ] No tools, specialists, engines, production checkpoint tasks, training or fine-tuning were run. Runtime probes remain simulations.- [ ] Confirm the `generation_hardening` block in `execution_configuration.json` and that no row has `finish_reason` of `length` or `repetition_loop`.
- [ ] Re-run 3 times and on an unseen prompt set before claiming generalisation.
- [ ] Dataset attached is the v2.5.0 input bundle (all pinned hashes and the input manifest verified; runtime regression tests present).


In [ ]:
from IPython.display import FileLink, display

display(FileLink(str(zip_path)))

In [ ]:
from pathlib import Path
from IPython.display import FileLink, display

# Locate the ZIP created by Cell 22
zip_file = Path(zip_path)

if not zip_file.is_file():
    raise FileNotFoundError(
        f"Evaluation ZIP not found: {zip_file}\n"
        "Run Cell 22 again and check its output."
    )

# Kaggle FileLink should use a path relative to /kaggle/working
relative_path = zip_file.relative_to("/kaggle/working")

print(f"ZIP found: {zip_file.name}")
print(f"Size: {zip_file.stat().st_size / (1024 * 1024):.2f} MB")

display(FileLink(str(relative_path)))